In [1]:
"""
B3. Unitary V_r verification, Chebyshev comparison, and worst-case analysis (CACHED)
-----------------------------------------------------------------------------------

This script verifies and benchmarks the implementation of the unitary V_r by
simulating full statevectors and comparing them against an analytical reference.

It uses the caching in unitary_vr.py: for each fixed (n, p), the θ(j) lookup
table and XOR-loader unitary are independent of r. The cache is optionally
warmed once per (n, p) before sweeping r so subsequent circuit builds reuse
the cached loader and skip table/unitary reconstruction.

For each parameter triple (n, p, r), the script:
    - builds the reduced register layout (j, theta, tgt),
    - constructs U_vr_circuit_no_x with theta uncomputation enabled,
    - simulates the circuit to obtain a full statevector,
    - builds an ideal reference statevector supported on the ancilla-zero subspace,
    - aligns global phase and computes ℓ2 statevector error and fidelity,
    - plots output amplitudes against Chebyshev polynomials T_r(x_j),
    - saves datasets and figures to disk,
    - extracts the top-k worst cases by error and exports them as JSON.

Printed output:
    - Per-(n, p, r) timing lines (build/simulate/verify/plot/total).
    - Per-(n, p) cache warm-up timing lines (if enabled).
    - A total runtime summary for the full sweep.
    - An ASCII table of the top-20 worst cases (by ℓ2 error).
    - A final total script runtime line.

Saved output:
    - Figures: PNG and EPS under figures/B3_unitary_vr/.
"""

import _init_path
import itertools
import json
from contextlib import contextmanager
from pathlib import Path
from typing import Tuple, Any, List
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector

from quantum.unitary_vr import (
    theta_to_fixed_bits,
    U_vr_circuit_no_x,
    get_theta_loader_gate,       # NEW: cache warm-up hook
    clear_theta_loader_cache,    # optional
)

from utils import _apply_publication_rcparams, _force_opaque


# =============================================================================
# Paths
# =============================================================================
FIG_DIR = Path("figures") / "B3_unitary_vr"


def _ensure_dirs() -> None:
    """ Ensure required output directories exist.

        This routine checks that the project-level 'figures/' directory exists
        and then ensures that the script-specific figure and data directories
        are created:

            figures/B3_unitary_vr/

        If the root 'figures/' directory is missing, the routine raises an
        error rather than silently creating it, matching the script's expected
        project layout.

        Args:
            None.

        Returns:
            None.

        Raises:
            FileNotFoundError: If the root 'figures/' directory does not exist.
    """
    root = Path("figures")
    if not root.exists():
        raise FileNotFoundError("Create 'figures/' directory first (e.g., `mkdir figures`).")
    FIG_DIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Layout labeling
# =============================================================================
def _layout_label_from_r(r: int) -> str:
    """ Map the Chebyshev order r to a human-readable layout label.

        This routine returns a short label used in saved datasets and tables.
        The label distinguishes the constant-case construction (r = 0) from
        the general construction (r ≥ 1).

        Args:
            r: Chebyshev polynomial order.

        Returns:
            A string label: "r=0" if r equals 0, otherwise "r>=1".

        Raises:
            None.
    """
    return "r=0" if int(r) == 0 else "r>=1"


# =============================================================================
# Fixed-point helper (needed for the analytical reference)
# =============================================================================
def bits_to_value(bits, p: int) -> float:
    """Convert a fixed-point bitstring into its real-valued angle estimate.

    This routine interprets a (p+2)-bit fixed-point representation with weights
    
        2^1, 2^0, 2^-1, … , 2^-p
    
    and returns the corresponding real value. The input bits are assumed to be
    ordered from most-significant to least-significant according to these weights.
    The function is used to reconstruct the quantized θ̂(j) in the analytical
    reference statevector.
    
    Args:
        bits: Iterable of bits representing the fixed-point expansion. Must
            contain exactly p+2 bits corresponding to weights
            (2^1, 2^0, 2^-1, …, 2^-p).
        p: Number of fractional bits in the representation.
    
    Returns:
        The real value represented by the fixed-point bitstring.
    
    Raises:
        ValueError: If the number of provided bits is not equal to p + 2.
    """
    bits = list(bits)
    if len(bits) != p + 2:
        raise ValueError(f"Expected {p+2} bits, got {len(bits)}")
    ks = [1, 0] + list(range(-1, -p - 1, -1))
    return sum(float(b) * (2.0 ** k) for b, k in zip(bits, ks))


# =============================================================================
# Verifier (settings)
# =============================================================================
QISKIT_MATCHES_PAPER_RX = True


# =============================================================================
# Circuit build + simulate once per (n,p,r)
# =============================================================================
def build_uvr_circuit(n: int, p: int, r: int) -> QuantumCircuit:
    """ Construct the reduced-register circuit implementing U_vr.

        This routine allocates the reduced register layout (j, theta, tgt)
        and appends the operations produced by U_vr_circuit_no_x with θ-register
        uncomputation enabled. It also validates the final qubit count.

        Args:
            n: Number of qubits in the index register j.
            p: Number of fractional bits for the fixed-point angle register.
            r: Chebyshev polynomial order.

        Returns:
            A QuantumCircuit over registers (j, theta, tgt) implementing U_vr.

        Raises:
            ValueError: If the constructed circuit qubit count does not match
                the expected n + (p+2) + 1 layout.
    """
    m = p + 2

    j = QuantumRegister(n, "j")
    th = QuantumRegister(m, "theta")
    tgt = QuantumRegister(1, "tgt")

    qc = QuantumCircuit(j, th, tgt)

    U_vr_circuit_no_x(
        n=n,
        p=p,
        r=r,
        qiskit_matches_paper_rx=QISKIT_MATCHES_PAPER_RX,
        uncompute_theta=True,
        j=j,
        th=th,
        tgt=tgt,
        qc=qc,
    )

    full_qubits = n + m + 1
    if qc.num_qubits != full_qubits:
        raise ValueError(
            f"Unexpected qubit count qc.num_qubits={qc.num_qubits}. Expected {full_qubits}."
        )
    return qc


def simulate_statevector(qc: QuantumCircuit) -> np.ndarray:
    """ Simulate a circuit and return its full statevector.

        This routine uses Qiskit's Statevector simulator to compute the
        statevector resulting from applying the provided circuit to |0…0⟩.

        Args:
            qc: QuantumCircuit to simulate.

        Returns:
            A complex NumPy array containing the statevector amplitudes.

        Raises:
            None.
    """
    return Statevector.from_instruction(qc).data


# =============================================================================
# Analytical reference state (full dim, but only theta=0 subspace populated)
# =============================================================================
def ideal_statevector(n: int, p: int, r: int) -> np.ndarray:
    """ Construct the exact mathematical target statevector.
        This uses the exact angle theta = arccos(x_j), not the 
        quantized angle used by the circuit. The simulated circuit 
        uses fixed-point encoded angles (p bits), so comparing the 
        simulated state to this reference intentionally measures 
        the approximation error introduced by angle quantization.

        This routine builds an analytical reference statevector for comparison
        with the simulated circuit output. The vector is allocated in the full
        Hilbert space of (j, theta, tgt), but it populates only the θ = 0
        subspace, consistent with running U_vr_circuit_no_x with θ uncomputation.

        For r = 0, the reference uses the constant-case rotation angle applied
        by the circuit.

        Args:
            n: Number of qubits in the index register j.
            p: Number of fractional bits for the fixed-point angle register.
            r: Chebyshev polynomial order.

        Returns:
            A complex NumPy array of length 2^(n + (p+2) + 1) representing the
            ideal reference statevector.

        Raises:
            None.
    """
    N = 2**n
    m = p + 2
    full_qubits = n + m + 1
    dim = 2**full_qubits

    ideal = np.zeros(dim, dtype=complex)

    # Register ordering (j, theta, tgt) and Qiskit's little-endian convention:
    # tgt is the most significant among these, at bit index (n + m).
    tgt_bit = 1 << (n + m)

    if r == 0:
        phi = -2.0 * np.pi / 3.0
        a0 = np.cos(phi / 2) / np.sqrt(N)
        a1 = (-1j * np.sin(phi / 2)) / np.sqrt(N)

        for jval in range(N):
            base = jval  # theta=0, tgt=0
            ideal[base] = a0
            ideal[base + tgt_bit] = a1
        return ideal

    for jval in range(N):
        xj = 2.0 * jval / N - 1.0
        theta = float(np.arccos(np.clip(xj, -1.0, 1.0)))
        #th_bits = theta_to_fixed_bits(theta, p)
        #th_hat = bits_to_value(th_bits, p)

        angle_paper = -r * theta
        phi = 2.0 * angle_paper if QISKIT_MATCHES_PAPER_RX else angle_paper

        a0 = np.cos(phi/2) / np.sqrt(N)
        a1 = (-1j * np.sin(phi/2)) / np.sqrt(N)

        base = jval  # theta=0, tgt=0 (after uncompute_theta=True)
        ideal[base] = a0
        ideal[base + tgt_bit] = a1

    return ideal


def verify_against_ideal(sv: np.ndarray, n: int, p: int, r: int) -> Tuple[float, float]:
    """ Verify a simulated statevector against the analytical reference.

        This routine constructs the ideal reference statevector, aligns the
        simulated statevector by global phase using the overlap with the ideal,
        and then computes:

            - ℓ2 error: || sv_aligned − ideal ||_2
            - fidelity: |⟨ideal, sv_aligned⟩|^2

        Args:
            sv: Simulated statevector to verify.
            n: Number of qubits in the index register j.
            p: Number of fractional bits for the fixed-point angle register.
            r: Chebyshev polynomial order.

        Returns:
            A tuple (err, fid) where err is the ℓ2 error and fid is the fidelity.

        Raises:
            None.
    """
    ideal = ideal_statevector(n, p, r)

    phase = np.vdot(ideal, sv)
    if abs(phase) > 0:
        sv_aligned = sv * np.exp(-1j * np.angle(phase))
    else:
        sv_aligned = sv

    err = np.linalg.norm(sv_aligned - ideal)
    fid = abs(np.vdot(ideal, sv_aligned)) ** 2
    return float(err), float(fid)


# =============================================================================
# Chebyshev utilities
# =============================================================================
def chebyshev_T_on_grid(n: int, r: int) -> Tuple[np.ndarray, np.ndarray]:
    """Evaluate the Chebyshev polynomial T_r on the uniform grid x_j.

    For the special case r = 0, this routine returns (1/2) * T_0(x)
    so verification plots show half the first Chebyshev polynomial.

    x_j = 2j / N − 1,   N = 2^n,   j = 0, …, N−1
    T_r(x) = cos(r arccos(x))
    """

    N = 2**n
    j = np.arange(N, dtype=int)
    x = 2.0 * j / N - 1.0

    if r == 0:
        T = 0.5 * np.ones(N)
        return j, T

    T = np.cos(r * np.arccos(np.clip(x, -1.0, 1.0)))
    return j, T


def extract_amp0(sv: np.ndarray, n: int) -> np.ndarray:
    """ Extract the scaled θ=0, tgt=0 amplitudes as a length-N vector.

        This routine assumes the reduced-register ordering (j, theta, tgt)
        and that the reference subspace corresponds to theta=0 and tgt=0.
        It returns the first N amplitudes of the statevector, scaled by √N,
        matching the plotting convention used for Chebyshev comparisons.

        Args:
            sv: Full statevector over (j, theta, tgt).
            n: Number of qubits in the index register, defining N = 2^n.

        Returns:
            A complex NumPy array of length N containing √N-scaled amplitudes.

        Raises:
            None.
    """
    N = 2**n
    return np.sqrt(N) * sv[:N]


# =============================================================================
# Plot output vs Chebyshev (consumes sv; no simulation inside)
# =============================================================================
def plot_output_vs_chebyshev_from_sv(sv: np.ndarray, n: int, p: int, r: int) -> None:
    """Plot output amplitudes versus Chebyshev values and save figure/data files."""
    _apply_publication_rcparams()

    N = 2**n
    j, T = chebyshev_T_on_grid(n, r)
    x = 2.0 * j / N - 1.0
    amps = extract_amp0(sv, n)
    amps_re = np.real(amps)

    tag = f"n{n}_p{p}_r{r}"

    fig, ax = plt.subplots(figsize=(7.4, 4.8), constrained_layout=True)

    # Colors
    cheb_color = "tab:blue"
    output_color = "tab:orange"

    # Styles
    predicted_style = dict(
        linestyle=":",
        marker="o",
        linewidth=1.8,
        markersize=4.5,
    )
    simulated_style = dict(
        linestyle="-",
        marker="o",
        linewidth=1.8,
        markersize=4.5,
    )

    ax.plot(
    x,
    T,
    color=cheb_color,
    label=rf"Predicted",
    **predicted_style,
    )

    ax.plot(
        x,
        amps_re,
        color=output_color,
        label=r"Simulated",
        **simulated_style,
    )

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(rf"$T_{{{int(r)}}}(x)$")

    step = 0.025
    ax.set_xlim(-1.0-step, 1.0+step)
    ax.set_xticks([-1.0, -0.5, 0.0, 0.5, 1.0])

    leg = ax.legend(
        loc="upper right",
        bbox_to_anchor=(0.98, 0.98),
        fontsize=9,
        handlelength=2.0,
        labelspacing=0.3,
        borderpad=0.3,
        handletextpad=0.4,
        borderaxespad=0.2,
        frameon=True,
    )
    
    if leg is not None and leg.get_frame() is not None:
        leg.get_frame().set_alpha(1.0)

    _force_opaque(ax)

    base = FIG_DIR / f"output_vs_chebyshev_{tag}"
    fig.savefig(str(base) + ".png", transparent=False, dpi=300)
    fig.savefig(str(base) + ".eps", format="eps", transparent=False)

    plt.close(fig)

# =============================================================================
# Worst-case extraction
# =============================================================================
def extract_worst_cases(df: pd.DataFrame, top_k: int = 20) -> pd.DataFrame:
    """ Extract the top-k worst cases by verification error.

        This routine removes rows with missing error or fidelity values and
        returns the top-k rows sorted by descending ℓ2 error.

        Args:
            df: DataFrame containing per-point verification results.
            top_k: Number of worst cases to return.

        Returns:
            A DataFrame containing the top-k rows by descending error.

        Raises:
            None.
    """
    df_ok = df.dropna(subset=["err", "fid"]).copy()
    return df_ok.sort_values("err", ascending=False).head(top_k).reset_index(drop=True)


# =============================================================================
# ASCII table printing
# =============================================================================
def _fmt_float(x: Any, ndigits: int = 10) -> str:
    """ Format a value as a fixed-precision floating-point string.

        This routine converts the input to float when possible and formats it
        with a fixed number of digits after the decimal point. If conversion
        fails, it falls back to string formatting. A None input yields an empty
        string.

        Args:
            x: Value to format.
            ndigits: Number of digits after the decimal point.

        Returns:
            A formatted string representation of x.

        Raises:
            ValueError: If ndigits is negative.
    """
    if ndigits < 0:
        raise ValueError("ndigits must be >= 0.")
    if x is None:
        return ""
    try:
        return f"{float(x): .{ndigits}f}"
    except Exception:
        return str(x)


def _fmt_int(x: Any) -> str:
    """ Format a value as an integer string when possible.

        This routine converts the input to int when possible. If conversion
        fails, it falls back to string formatting. A None input yields an empty
        string.

        Args:
            x: Value to format.

        Returns:
            A formatted string representation of x.

        Raises:
            None.
    """
    if x is None:
        return ""
    try:
        return str(int(x))
    except Exception:
        return str(x)

def print_worst_cases_table(worst: "pd.DataFrame | List[dict]") -> None:
    """Print an ASCII table of worst-case verification results (no layout column)."""
    if isinstance(worst, pd.DataFrame):
        rows = worst.to_dict(orient="records")
    elif isinstance(worst, list) and all(isinstance(r, dict) for r in worst):
        rows = worst
    else:
        raise ValueError("worst must be a pandas DataFrame or a list of dictionaries.")

    headers = ["#", "n", "p", "r", "err", "fid", "qubits", "depth", "size"]
    aligns  = ["R", "R", "R", "R", "R",  "R",   "R",     "R",     "R"]

    table_rows: List[List[str]] = []
    for i, r_ in enumerate(rows, 1):
        table_rows.append(
            [
                str(i),
                _fmt_int(r_.get("n")),
                _fmt_int(r_.get("p")),
                _fmt_int(r_.get("r")),
                _fmt_float(r_.get("err"), 10),
                _fmt_float(r_.get("fid"), 10),
                _fmt_int(r_.get("qubits")),
                _fmt_int(r_.get("depth")),
                _fmt_int(r_.get("size")),
            ]
        )

    widths = [len(h) for h in headers]
    for row in table_rows:
        for j, cell in enumerate(row):
            widths[j] = max(widths[j], len(cell))

    TL, TR, BL, BR = "┌", "┐", "└", "┘"
    H, V = "─", "│"
    TJ, BJ, LJ, RJ, CJ = "┬", "┴", "├", "┤", "┼"

    def hline(left: str, mid: str, right: str) -> str:
        return left + mid.join(H * (w + 2) for w in widths) + right

    def format_cell(text: str, w: int, align: str, header: bool = False) -> str:
        if header:
            return f" {text:^{w}} "
        if align == "R":
            return f" {text:>{w}} "
        return f" {text:<{w}} "

    def render_row(row: List[str], header: bool = False) -> str:
        return V + V.join(
            format_cell(row[j], widths[j], aligns[j], header=header)
            for j in range(len(headers))
        ) + V

    print(hline(TL, TJ, TR))
    print(render_row(headers, header=True))
    print(hline(LJ, CJ, RJ))
    for row in table_rows:
        print(render_row(row))
    print(hline(BL, BJ, BR))


# =============================================================================
# Runner
# =============================================================================
@contextmanager
def timer():
    """ Context manager for timing code blocks.

        This routine yields a callable that returns the elapsed time in seconds
        since entering the context, measured using time.perf_counter().

        Example usage:

            with timer() as t:
                ... work ...
            elapsed = t()

        Args:
            None.

        Returns:
            A context manager that yields a zero-argument callable returning
            elapsed seconds as a float.

        Raises:
            None.
    """
    t0 = time.perf_counter()
    try:
        yield lambda: time.perf_counter() - t0
    finally:
        pass


def run_tests(n_values, p_values, r_values, *, warm_cache_per_np: bool = True) -> pd.DataFrame:
    """ Execute verification tests over parameter grids.

        This routine runs the full verification pipeline over all combinations
        of (n, p, r) drawn from the provided iterables. For each point it:

            - builds the circuit implementing U_vr,
            - simulates the full statevector,
            - verifies against the analytical ideal (error and fidelity),
            - generates and saves Chebyshev comparison plots,
            - records timing, circuit statistics, and verification metrics.

        Caching behavior:
            If warm_cache_per_np is True and there are multiple r values for a
            fixed (n, p), the routine calls get_theta_loader_gate(n, p) once
            per (n, p) prior to sweeping r. This moves the one-time loader
            construction cost into an explicit warm-up line so subsequent
            per-point build timings reflect reuse of the cached loader.

        If there is only one r value, explicit warm-up is skipped even when
        warm_cache_per_np is True, since in that case warming would only
        shift the first-build cost rather than reduce total work.

        Args:
            n_values: Iterable of n values defining N = 2^n grid sizes.
            p_values: Iterable of p values defining (p+2) fixed-point bits.
            r_values: Iterable of Chebyshev orders r to test.
            warm_cache_per_np: Whether to warm the (n, p) cache once before sweeping r.

        Returns:
            A DataFrame containing the top-20 worst cases by error.

        Raises:
            FileNotFoundError: If the required root figures directory is missing.
    """
    _ensure_dirs()

    n_values = tuple(n_values)
    p_values = tuple(p_values)
    r_values = tuple(r_values)

    rows = []

    clear_theta_loader_cache()

    do_explicit_warmup = warm_cache_per_np and (len(r_values) > 1)

    for n in n_values:
        for p in p_values:
            warm_s = 0.0
            first_r_for_np = True

            if do_explicit_warmup:
                t_warm0 = time.perf_counter()
                _ = get_theta_loader_gate(n, p)
                warm_s = time.perf_counter() - t_warm0
                print(f"(n={n}, p={p}) warmup={warm_s:.3f}s")

            for r in r_values:
                t_point0 = time.perf_counter()

                # Build
                t0 = time.perf_counter()
                qc = build_uvr_circuit(n, p, r)
                build_s = time.perf_counter() - t0

                # Circuit metadata
                qubits = qc.num_qubits
                depth = qc.depth()
                size = qc.size()

                # Simulate
                t0 = time.perf_counter()
                sv = simulate_statevector(qc)
                sim_s = time.perf_counter() - t0

                # Verify
                t0 = time.perf_counter()
                err, fid = verify_against_ideal(sv, n, p, r)
                verify_s = time.perf_counter() - t0

                # Plot
                t0 = time.perf_counter()
                plot_output_vs_chebyshev_from_sv(sv, n, p, r)
                plot_s = time.perf_counter() - t0

                point_s = time.perf_counter() - t_point0

                warmup_this_row = warm_s if first_r_for_np else 0.0
                accounted_total_s = warmup_this_row + point_s

                print(
                    f"(n={n}, p={p}, r={r}) "
                    f"warmup={warmup_this_row:.3f}s "
                    f"build={build_s:.3f}s "
                    f"sim={sim_s:.3f}s "
                    f"verify={verify_s:.3f}s "
                    f"plot={plot_s:.3f}s "
                    f"total={accounted_total_s:.3f}s"
                )

                rows.append({
                    "n": n,
                    "p": p,
                    "r": r,
                    "qubits": qubits,
                    "depth": depth,
                    "size": size,
                    "warmup_s": warmup_this_row,
                    "build_s": build_s,
                    "sim_s": sim_s,
                    "verify_s": verify_s,
                    "plot_s": plot_s,
                    "point_total_s": point_s,
                    "accounted_total_s": accounted_total_s,
                    "err": err,
                    "fid": fid,
                })

                del qc, sv
                first_r_for_np = False

    df = pd.DataFrame(rows)

    if not df.empty:
        print(f"\nSUM OF ACCOUNTED TOTALS: {df['accounted_total_s'].sum():.3f}s")

    return df

Repo root: /Users/junaida/Documents/Non-Uniform-QFT
Added to sys.path: /Users/junaida/Documents/Non-Uniform-QFT/src


In [2]:
if __name__ == "__main__":
    total_start = time.perf_counter()

    worst_df = run_tests(
        n_values=range(4, 7),
        p_values=range(5,6),
        r_values=range(2, 5),
        warm_cache_per_np=False,
    )

    print("\nTOP 20 WORST CASES (by error)")
    print_worst_cases_table(worst_df)

    total_runtime = time.perf_counter() - total_start
    print(f"\nTOTAL SCRIPT RUNTIME: {total_runtime:.3f} seconds")

(n=4, p=5, r=2) warmup=0.000s build=0.519s sim=0.132s verify=0.000s plot=0.527s total=1.207s
(n=4, p=5, r=3) warmup=0.000s build=0.036s sim=0.107s verify=0.000s plot=0.364s total=0.528s
(n=4, p=5, r=4) warmup=0.000s build=0.032s sim=0.104s verify=0.000s plot=0.368s total=0.524s
(n=5, p=5, r=2) warmup=0.000s build=3.495s sim=0.750s verify=0.000s plot=0.368s total=4.787s
(n=5, p=5, r=3) warmup=0.000s build=0.226s sim=0.662s verify=0.000s plot=0.382s total=1.464s
(n=5, p=5, r=4) warmup=0.000s build=0.253s sim=0.645s verify=0.000s plot=0.402s total=1.507s
(n=6, p=5, r=2) warmup=0.000s build=26.017s sim=4.579s verify=0.001s plot=0.393s total=31.951s
(n=6, p=5, r=3) warmup=0.000s build=1.192s sim=4.477s verify=0.001s plot=0.403s total=6.932s
(n=6, p=5, r=4) warmup=0.000s build=1.218s sim=4.326s verify=0.001s plot=0.405s total=6.827s

SUM OF ACCOUNTED TOTALS: 55.727s

TOP 20 WORST CASES (by error)
┌───┬───┬───┬───┬───────────────┬───────────────┬────────┬───────┬──────┐
│ # │ n │ p │ r │     

In [3]:
if __name__ == "__main__":
    total_start = time.perf_counter()

    worst_df = run_tests(
        n_values=range(4, 5),
        p_values=range(1,9),
        r_values=range(4, 5),
        warm_cache_per_np=True,
    )

    print("\nTOP 20 WORST CASES (by error)")
    print_worst_cases_table(worst_df)

    total_runtime = time.perf_counter() - total_start
    print(f"\nTOTAL SCRIPT RUNTIME: {total_runtime:.3f} seconds")

(n=4, p=1, r=4) warmup=0.000s build=0.003s sim=0.002s verify=0.000s plot=0.381s total=0.386s
(n=4, p=2, r=4) warmup=0.000s build=0.003s sim=0.002s verify=0.000s plot=0.366s total=0.372s
(n=4, p=3, r=4) warmup=0.000s build=0.016s sim=0.007s verify=0.000s plot=0.369s total=0.394s
(n=4, p=4, r=4) warmup=0.000s build=0.085s sim=0.023s verify=0.000s plot=0.366s total=0.478s
(n=4, p=5, r=4) warmup=0.000s build=0.493s sim=0.109s verify=0.001s plot=0.363s total=0.985s
(n=4, p=6, r=4) warmup=0.000s build=3.366s sim=0.598s verify=0.000s plot=0.388s total=4.544s
(n=4, p=7, r=4) warmup=0.000s build=27.592s sim=4.376s verify=0.001s plot=0.390s total=33.445s
(n=4, p=8, r=4) warmup=0.000s build=207.735s sim=24.364s verify=0.001s plot=0.397s total=240.059s

SUM OF ACCOUNTED TOTALS: 280.664s

TOP 20 WORST CASES (by error)
┌───┬───┬───┬───┬───────────────┬───────────────┬────────┬───────┬──────┐
│ # │ n │ p │ r │      err      │      fid      │ qubits │ depth │ size │
├───┼───┼───┼───┼───────────────┼──